# Agent Memory: Short-Term — Conversation and Working Memory

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/10_agent_memory_short_term.ipynb)

## What This Notebook Teaches You

Agents that hold conversations face a fundamental constraint: **LLM context windows are finite**, but conversations can be arbitrarily long. An agent that naively appends every message will eventually exceed its context window and either crash or silently lose information.

This notebook tackles the **short-term memory problem** — how to maintain useful conversation context within a single session.

By the end, you will:

1. **Understand the memory problem** — why naive full-history fails at scale
2. **Implement 4 memory strategies** — full history, sliding window, summarization, importance-weighted
3. **Build a MemoryManager** — that tracks tokens and triggers compression automatically
4. **Stress-test all strategies** — 30-turn simulation measuring token usage and information retention
5. **Build a Working Memory** — a scratchpad pattern for agents to store key findings

Every strategy is implemented from scratch with measurable trade-offs.

---

> **Prerequisites**: Notebooks 01–09 (agent fundamentals, tool use, ReAct).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~45–60 minutes.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What This Notebook Teaches You**
- Understand **: Environment Setup**
- Understand **The Memory Problem**
- Understand **Strategy 1: Full History Memory**
- Understand **Strategy 2: Sliding Window Memory**


## Part 0: Environment Setup

We load Qwen3-8B with 4-bit quantization. Short-term memory is purely about managing the conversation messages sent to the model.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. The Memory Problem

### Why Memory Matters

Consider an agent helping a user debug a complex issue across 30+ turns. Without memory management:

| Turn | Tokens (cumulative) | Problem |
|------|---------------------|---------|
| 1 | ~200 | Fine |
| 10 | ~3,000 | Fine |
| 20 | ~8,000 | Slowing down |
| 30 | ~15,000 | Approaching context limit |
| 50 | ~30,000 | **Exceeds context window** |

The agent needs to remember what matters while fitting within its context budget. This is exactly the challenge human working memory solves — we don't remember every word of a conversation, but we retain the gist and key facts.

### Token Counting Utility

Let's build a helper to measure how many tokens a conversation consumes.

In [ ]:
def count_tokens(messages):
    """Count tokens in a message list using the model's tokenizer."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
    )
    return len(tokenizer.encode(text))

def format_messages_preview(messages, max_chars=120):
    """Show a compact preview of messages."""
    lines = []
    for m in messages:
        role = m["role"]
        content = m["content"][:max_chars].replace("\n", " ")
        if len(m["content"]) > max_chars:
            content += "..."
        lines.append(f"  [{role}] {content}")
    return "\n".join(lines)

# Demonstrate token growth
sample_messages = [{"role": "system", "content": "You are a helpful assistant."}]
print(f"System message alone: {count_tokens(sample_messages)} tokens")

for i in range(1, 31):
    sample_messages.append({"role": "user", "content": f"Turn {i}: Tell me about topic number {i} in detail."})
    sample_messages.append({"role": "assistant", "content": f"Here is information about topic {i}. " * 10})

    if i in [1, 5, 10, 15, 20, 25, 30]:
        tokens = count_tokens(sample_messages)
        print(f"After {i:2d} turns: {tokens:6,} tokens ({len(sample_messages):3d} messages)")

print("\n⚠️  Token growth is roughly linear — eventually exceeds any context window.")

## 2. Strategy 1: Full History Memory

The simplest approach: keep everything. This is what most chatbot demos do.

**Pros**: Perfect information retention — nothing is ever lost.
**Cons**: Unbounded token growth. Will eventually exceed context window.

This serves as our **baseline** for comparison.

In [ ]:
class FullHistoryMemory:
    """Keep all messages forever. Simple but unbounded growth."""

    def __init__(self, system_prompt="You are a helpful assistant."):
        self.system_prompt = system_prompt
        self.messages = []

    def add_message(self, role, content):
        self.messages.append({"role": role, "content": content})

    def get_context(self):
        """Return full message history with system prompt."""
        return [{"role": "system", "content": self.system_prompt}] + self.messages

    def token_count(self):
        return count_tokens(self.get_context())

    def message_count(self):
        return len(self.messages)

    def reset(self):
        self.messages = []

    def __repr__(self):
        return f"FullHistoryMemory({self.message_count()} msgs, {self.token_count()} tokens)"

# Test it
memory = FullHistoryMemory("You are a coding assistant.")
memory.add_message("user", "How do I read a file in Python?")
memory.add_message("assistant", "Use open('file.txt', 'r') to read a file. The with statement ensures proper cleanup.")
memory.add_message("user", "What about binary files?")
memory.add_message("assistant", "Use 'rb' mode: open('file.txt', 'rb'). This returns bytes instead of str.")

print(memory)
print(f"\nContext preview:")
print(format_messages_preview(memory.get_context()))

In [ ]:
# Demonstrate token growth with FullHistoryMemory over 30 turns
fh_memory = FullHistoryMemory("You are a helpful assistant.")
growth_data = []

for i in range(1, 31):
    fh_memory.add_message("user", f"Question {i}: Explain concept number {i} in machine learning, including examples and practical applications.")
    fh_memory.add_message("assistant", f"Concept {i} is an important topic. Here are the key points: " +
                          f"First, the definition involves... Second, the applications include... " * 5 +
                          f"In practice, you would use concept {i} when...")
    tokens = fh_memory.token_count()
    growth_data.append((i, tokens))

print("Full History Memory — Token Growth Over 30 Turns")
print("=" * 55)
print(f"{'Turn':>5} {'Tokens':>8} {'Growth':>10}  Bar")
print("-" * 55)

prev = 0
for turn, tokens in growth_data:
    growth = tokens - prev
    bar = "█" * (tokens // 200)
    if turn % 3 == 0 or turn <= 3 or turn == 30:
        print(f"{turn:5d} {tokens:8,} {'+' + str(growth):>10}  {bar}")
    prev = tokens

print(f"\n📊 Final: {growth_data[-1][1]:,} tokens after 30 turns")
print(f"   Average per turn: {growth_data[-1][1] // 30:,} tokens")

## 3. Strategy 2: Sliding Window Memory

Keep only the last **N** message pairs. Old messages are simply dropped.

**Pros**: Bounded token usage. Simple to implement.
**Cons**: Hard cutoff — information older than the window is completely lost.

This is the approach used by many production chatbots. The key parameter is the **window size**.

In [ ]:
class SlidingWindowMemory:
    """Keep only the last window_size user-assistant pairs."""

    def __init__(self, system_prompt="You are a helpful assistant.", window_size=10):
        self.system_prompt = system_prompt
        self.window_size = window_size
        self.messages = []

    def add_message(self, role, content):
        self.messages.append({"role": role, "content": content})

    def get_context(self):
        """Return system prompt + last window_size pairs of messages."""
        # Keep last window_size * 2 messages (user + assistant pairs)
        max_messages = self.window_size * 2
        recent = self.messages[-max_messages:] if len(self.messages) > max_messages else self.messages
        return [{"role": "system", "content": self.system_prompt}] + recent

    def token_count(self):
        return count_tokens(self.get_context())

    def total_messages_seen(self):
        return len(self.messages)

    def messages_in_context(self):
        max_messages = self.window_size * 2
        return min(len(self.messages), max_messages)

    def reset(self):
        self.messages = []

    def __repr__(self):
        return (f"SlidingWindowMemory(window={self.window_size}, "
                f"in_context={self.messages_in_context()}, "
                f"total_seen={self.total_messages_seen()}, "
                f"{self.token_count()} tokens)")

# Compare different window sizes
print("Sliding Window — Different Window Sizes After 30 Turns")
print("=" * 60)

for window_size in [3, 5, 10, 15]:
    sw_memory = SlidingWindowMemory("You are a helpful assistant.", window_size=window_size)

    for i in range(1, 31):
        sw_memory.add_message("user", f"Question {i}: Explain concept number {i} in ML.")
        sw_memory.add_message("assistant", f"Concept {i} is important. Key points: " +
                              f"First, the definition... Second, applications... " * 5 +
                              f"In practice, concept {i} is used when...")

    print(f"  Window={window_size:2d}: {sw_memory.token_count():6,} tokens "
          f"({sw_memory.messages_in_context()} msgs in context, "
          f"{sw_memory.total_messages_seen()} total seen)")

print(f"\n  Full:    {growth_data[-1][1]:6,} tokens (all 60 msgs in context)")
print(f"\n📊 Sliding window bounds memory regardless of conversation length.")

## 4. Strategy 3: Summarization Memory

Use the LLM itself to **compress old messages into a summary**, then keep the summary + recent messages.

**Pros**: Retains key information from old messages while bounding token usage.
**Cons**: Summarization costs tokens, takes time, and may lose details. Requires careful prompting.

The approach:
1. Keep recent messages in full
2. When history exceeds a threshold, summarize older messages
3. Prepend the summary to the context

In [ ]:
class SummarizationMemory:
    """Summarize old messages to compress history while retaining key info."""

    def __init__(self, system_prompt="You are a helpful assistant.",
                 recent_window=5, summary_trigger=10):
        self.system_prompt = system_prompt
        self.recent_window = recent_window      # keep this many recent pairs
        self.summary_trigger = summary_trigger   # summarize when we exceed this many pairs
        self.messages = []
        self.summary = ""
        self.summaries_performed = 0
        self.total_messages_seen = 0

    def add_message(self, role, content):
        self.messages.append({"role": role, "content": content})
        self.total_messages_seen += 1

        # Check if we need to summarize
        num_pairs = len(self.messages) // 2
        if num_pairs >= self.summary_trigger:
            self._summarize()

    def _summarize(self):
        """Compress older messages into a summary, keeping recent ones."""
        keep_count = self.recent_window * 2
        old_messages = self.messages[:-keep_count]
        recent_messages = self.messages[-keep_count:]

        # Build text of old messages for summarization
        old_text = ""
        if self.summary:
            old_text += f"Previous summary:\n{self.summary}\n\n"
        old_text += "Messages to summarize:\n"
        for m in old_messages:
            old_text += f"[{m['role']}]: {m['content']}\n"

        # Use LLM to create summary
        summary_prompt = [
            {"role": "system", "content": "You are a precise summarizer. Summarize the conversation below, preserving ALL key facts, decisions, and technical details. Be concise but complete. Output only the summary."},
            {"role": "user", "content": old_text}
        ]

        self.summary = generate(summary_prompt, max_new_tokens=300, temperature=0.3)
        self.messages = recent_messages
        self.summaries_performed += 1
        print(f"  📝 Summarization #{self.summaries_performed}: compressed {len(old_messages)} msgs → {len(self.summary)} chars")

    def get_context(self):
        """Return system prompt + summary (if any) + recent messages."""
        context = [{"role": "system", "content": self.system_prompt}]
        if self.summary:
            context.append({
                "role": "system",
                "content": f"Summary of earlier conversation:\n{self.summary}"
            })
        context.extend(self.messages)
        return context

    def token_count(self):
        return count_tokens(self.get_context())

    def reset(self):
        self.messages = []
        self.summary = ""
        self.summaries_performed = 0
        self.total_messages_seen = 0

    def __repr__(self):
        return (f"SummarizationMemory(recent={len(self.messages)} msgs, "
                f"summary={'yes' if self.summary else 'no'}, "
                f"compressions={self.summaries_performed}, "
                f"{self.token_count()} tokens)")

# Test summarization memory
print("Summarization Memory — Demonstration")
print("=" * 50)

sm_memory = SummarizationMemory(
    system_prompt="You are a Python tutor.",
    recent_window=3,
    summary_trigger=6
)

conversations = [
    ("user", "What is a Python list?"),
    ("assistant", "A list is an ordered, mutable collection. Create with square brackets: my_list = [1, 2, 3]. Lists support indexing, slicing, and methods like append(), insert(), and remove()."),
    ("user", "How do I sort a list?"),
    ("assistant", "Two ways: list.sort() modifies in-place, sorted(list) returns a new sorted list. Both accept key= and reverse= parameters. Example: sorted(names, key=len, reverse=True)."),
    ("user", "What about list comprehensions?"),
    ("assistant", "List comprehensions create lists concisely: [x**2 for x in range(10) if x % 2 == 0]. They replace map+filter patterns and are generally faster than equivalent for loops."),
    ("user", "Tell me about dictionaries."),
    ("assistant", "Dicts map keys to values: d = {'name': 'Alice', 'age': 30}. Keys must be hashable. Methods: .keys(), .values(), .items(), .get(key, default). Dict comprehensions: {k: v for k, v in pairs}."),
    ("user", "How do I merge two dicts?"),
    ("assistant", "Python 3.9+: merged = dict1 | dict2. Earlier: merged = {**dict1, **dict2}. Or dict1.update(dict2) to modify in-place. The rightmost value wins for duplicate keys."),
]

for role, content in conversations:
    sm_memory.add_message(role, content)

print(f"\nAfter {len(conversations)} messages:")
print(sm_memory)

if sm_memory.summary:
    print(f"\n📝 Summary of old messages:")
    print(f"   {sm_memory.summary[:500]}")

# Add a few more to trigger another potential summarization
more_conversations = [
    ("user", "What about sets in Python?"),
    ("assistant", "Sets are unordered collections of unique elements: s = {1, 2, 3}. Support union (|), intersection (&), difference (-). Great for deduplication and membership testing (O(1) lookup)."),
    ("user", "And tuples?"),
    ("assistant", "Tuples are immutable sequences: t = (1, 2, 3). Used for fixed collections, dict keys, and function return values. Named tuples add field names: from collections import namedtuple."),
    ("user", "Summarize everything we've discussed about Python data structures."),
    ("assistant", "We covered: lists (ordered, mutable, comprehensions, sorting), dicts (key-value, merging, comprehensions), sets (unique, set operations), and tuples (immutable, named tuples). Each has specific use cases based on mutability, ordering, and uniqueness requirements."),
]

for role, content in more_conversations:
    sm_memory.add_message(role, content)

print(f"\nAfter {sm_memory.total_messages_seen} total messages:")
print(sm_memory)

## 5. Strategy 4: Importance-Weighted Memory

Not all messages are equally important. A tool result with factual data is more valuable than a casual greeting. This strategy **scores each message by importance** and keeps the highest-scoring ones.

**Scoring heuristics**:
- Tool results / code outputs → high importance
- Messages with facts, numbers, decisions → medium-high
- Questions and answers → medium
- Greetings, confirmations, filler → low

**Pros**: Retains the most valuable information within a token budget.
**Cons**: Importance scoring is heuristic — mistakes lose critical info.

In [ ]:
class ImportanceWeightedMemory:
    """Keep messages ranked by importance within a token budget."""

    def __init__(self, system_prompt="You are a helpful assistant.", max_tokens=3000):
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.messages = []  # list of (message_dict, importance_score, index)
        self._index = 0

    @staticmethod
    def score_importance(message):
        """Heuristic importance scoring for a message."""
        content = message["content"].lower()
        role = message["role"]
        score = 5.0  # baseline

        # Role-based adjustments
        if role == "system":
            return 10.0  # always keep system messages

        # Content signals — higher importance
        if any(kw in content for kw in ["error", "exception", "traceback", "bug"]):
            score += 3.0  # errors are critical context
        if any(kw in content for kw in ["result:", "output:", "found:", "answer:"]):
            score += 2.5  # tool results and findings
        if any(kw in content for kw in ["decision:", "conclusion:", "therefore", "we should"]):
            score += 2.0  # decisions and conclusions
        if any(kw in content for kw in ["def ", "class ", "import ", "```"]):
            score += 2.0  # code content
        if re.search(r'\d+\.\d+|\d{3,}', content):
            score += 1.5  # contains numbers/data

        # Content signals — lower importance
        if any(kw in content for kw in ["hello", "hi ", "thanks", "thank you", "ok", "sure"]):
            score -= 2.0
        if len(content) < 20:
            score -= 1.5  # very short messages are often low-value
        if content.startswith("yes") or content.startswith("no"):
            score -= 1.0

        # Length bonus (longer messages tend to have more info)
        score += min(len(content) / 500, 2.0)

        return max(score, 1.0)

    def add_message(self, role, content):
        msg = {"role": role, "content": content}
        importance = self.score_importance(msg)
        self.messages.append((msg, importance, self._index))
        self._index += 1

    def get_context(self):
        """Return system prompt + highest-importance messages within budget."""
        context = [{"role": "system", "content": self.system_prompt}]
        base_tokens = count_tokens(context)
        budget = self.max_tokens - base_tokens

        # Sort by importance (descending), break ties by recency
        ranked = sorted(self.messages, key=lambda x: (x[1], x[2]), reverse=True)

        selected = []
        used_tokens = 0

        for msg, importance, idx in ranked:
            msg_tokens = count_tokens([msg])
            if used_tokens + msg_tokens <= budget:
                selected.append((msg, idx))
                used_tokens += msg_tokens

        # Re-sort selected by original order (preserve conversation flow)
        selected.sort(key=lambda x: x[1])
        context.extend([msg for msg, idx in selected])

        return context

    def token_count(self):
        return count_tokens(self.get_context())

    def total_messages_seen(self):
        return len(self.messages)

    def messages_in_context(self):
        return len(self.get_context()) - 1  # subtract system prompt

    def reset(self):
        self.messages = []
        self._index = 0

    def __repr__(self):
        return (f"ImportanceWeightedMemory(budget={self.max_tokens}, "
                f"in_context={self.messages_in_context()}, "
                f"total_seen={self.total_messages_seen()}, "
                f"{self.token_count()} tokens)")

# Demonstrate importance scoring
test_messages = [
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi there!"},
    {"role": "user", "content": "I'm getting an error: TypeError: cannot unpack non-sequence NoneType"},
    {"role": "assistant", "content": "The error means a function returned None when you expected a tuple. Check if your function has a return statement on all code paths."},
    {"role": "user", "content": "ok"},
    {"role": "user", "content": "Result: The query returned 1,847 matching records with 99.2% precision"},
    {"role": "assistant", "content": "Decision: We should use the filtered dataset with the 1,847 records since 99.2% precision exceeds our 95% threshold."},
    {"role": "user", "content": "thanks"},
]

print("Importance Scoring Demonstration")
print("=" * 70)
print(f"{'Role':<12} {'Score':>6}  Content Preview")
print("-" * 70)
for msg in test_messages:
    score = ImportanceWeightedMemory.score_importance(msg)
    preview = msg["content"][:55] + ("..." if len(msg["content"]) > 55 else "")
    print(f"{msg['role']:<12} {score:6.1f}  {preview}")

print("\n✓ Errors, results, and decisions score highest; greetings score lowest.")

## 6. MemoryManager — Unified Memory Controller

In practice, you want a single interface that:
1. **Tracks token usage** as messages are added
2. **Triggers compression** when approaching the context limit
3. **Supports swappable strategies** — change memory behavior without changing agent code

This is the pattern used in production agent frameworks like LangChain and AutoGen.

In [ ]:
class MemoryManager:
    """Unified memory controller with pluggable strategies and token tracking."""

    def __init__(self, strategy, context_limit=4096, compression_threshold=0.75):
        """
        Args:
            strategy: Memory strategy instance (FullHistory, SlidingWindow, etc.)
            context_limit: Maximum tokens allowed in context
            compression_threshold: Trigger compression at this fraction of limit
        """
        self.strategy = strategy
        self.context_limit = context_limit
        self.compression_threshold = compression_threshold
        self.token_history = []    # track token usage over time
        self.compressions = 0

    def add_message(self, role, content):
        """Add a message and check if compression is needed."""
        self.strategy.add_message(role, content)
        tokens = self.strategy.token_count()
        self.token_history.append(tokens)

        # Check if we need to compress
        threshold = int(self.context_limit * self.compression_threshold)
        if tokens > threshold:
            self._handle_overflow(tokens)

    def _handle_overflow(self, current_tokens):
        """Handle when token count exceeds threshold."""
        strategy_name = type(self.strategy).__name__

        if strategy_name == "SummarizationMemory":
            # SummarizationMemory handles its own compression
            pass
        elif strategy_name == "SlidingWindowMemory":
            # Window already bounds it — just warn if window is too large
            if current_tokens > self.context_limit:
                print(f"  ⚠️  SlidingWindow exceeds limit even with window. Consider smaller window.")
        elif strategy_name == "FullHistoryMemory":
            print(f"  ⚠️  FullHistory at {current_tokens}/{self.context_limit} tokens. No compression available!")
        # ImportanceWeighted is self-bounding via max_tokens

        self.compressions += 1

    def get_context(self):
        return self.strategy.get_context()

    def get_stats(self):
        current = self.strategy.token_count()
        return {
            "strategy": type(self.strategy).__name__,
            "current_tokens": current,
            "context_limit": self.context_limit,
            "utilization": f"{current / self.context_limit * 100:.1f}%",
            "peak_tokens": max(self.token_history) if self.token_history else 0,
            "messages_added": len(self.token_history),
        }

    def print_stats(self):
        stats = self.get_stats()
        print(f"📊 {stats['strategy']}")
        print(f"   Tokens: {stats['current_tokens']:,} / {stats['context_limit']:,} ({stats['utilization']})")
        print(f"   Peak: {stats['peak_tokens']:,} | Messages added: {stats['messages_added']}")

# Quick demo
manager = MemoryManager(
    strategy=SlidingWindowMemory("You are a helpful assistant.", window_size=5),
    context_limit=4096,
    compression_threshold=0.75
)

for i in range(15):
    manager.add_message("user", f"Question {i+1} about an important topic with details.")
    manager.add_message("assistant", f"Answer {i+1} with detailed explanation. " * 8)

manager.print_stats()

## 7. 30-Turn Stress Test

Now we simulate a realistic 30-turn conversation and run it through all four strategies, measuring:

- **Token usage** — how many tokens each strategy uses at each turn
- **Information retention** — can the agent still reference facts from early turns?

The simulated conversation covers a multi-step debugging session where early facts are referenced later.

In [ ]:
# Simulate a 30-turn debugging conversation
conversation_turns = []

topics = [
    ("I'm building a REST API with FastAPI. It handles user authentication.", "Got it. FastAPI is a great choice for REST APIs. Key components: path operations, Pydantic models, dependency injection for auth."),
    ("The database is PostgreSQL, accessed via SQLAlchemy async.", "SQLAlchemy async with asyncpg driver. You'll want to use async sessions and the declarative base for models."),
    ("Users table has: id, email, hashed_password, is_active, created_at.", "Good schema. Make sure email has a unique constraint and consider adding an index on it for login queries."),
    ("I'm using bcrypt for password hashing with cost factor 12.", "Bcrypt with cost 12 is solid — about 250ms per hash. Good balance of security and performance."),
    ("JWT tokens for auth: access token (15min) + refresh token (7 days).", "Standard JWT setup. Store refresh tokens in httponly cookies, access tokens in memory. Consider token rotation on refresh."),
    ("Error: 'async_generator' object has no attribute 'execute'", "This means you're using 'async for' syntax where you need 'await'. Check if you're calling session.execute() correctly with 'await'."),
    ("Fixed it! Was missing 'await' on the session factory.", "Classic async pitfall. The session factory returns a coroutine, not a session directly. Always await it."),
    ("Now getting 401 on protected endpoints even with valid token.", "Debug steps: 1) Verify token generation, 2) Check token extraction from header, 3) Verify decode with same secret key."),
    ("The secret key was different between token creation and validation!", "Found it! Use a single config source for JWT_SECRET. Environment variable loaded once at startup."),
    ("Added rate limiting: 100 requests/minute per IP using slowapi.", "Good choice. Consider different limits for auth endpoints (stricter) vs data endpoints (more relaxed)."),
    ("Performance concern: each request does 3 DB queries.", "Optimize with: 1) Eager loading relationships, 2) Caching user sessions in Redis, 3) Connection pooling."),
    ("Added Redis caching. Response time dropped from 120ms to 45ms.", "Great improvement — 62.5% reduction. Monitor cache hit rate; aim for >90% on auth lookups."),
    ("Writing tests now. Using pytest-asyncio for async test support.", "Good testing setup. Use fixtures for DB sessions and test client. Consider factory_boy for test data."),
    ("Coverage at 73%. Missing edge cases in error handling.", "Target 80%+. Focus on: expired tokens, malformed requests, DB connection failures, concurrent access."),
    ("Deploying to AWS. Using ECS with Fargate.", "Fargate is good for containerized FastAPI. Set up ALB for HTTPS termination, health checks on /health endpoint."),
    ("Need to set up CI/CD. What's the best approach?", "GitHub Actions with: lint → test → build Docker → push ECR → deploy ECS. Add staging environment before prod."),
    ("Database migrations with Alembic. Had a conflict when two devs migrated simultaneously.", "Alembic conflicts: merge branches with 'alembic merge heads'. Prevent by: linear migration policy, CI checks for multiple heads."),
    ("Added OpenTelemetry tracing. Can see full request lifecycle now.", "Excellent observability. Trace across services: request → auth → DB → cache → response. Set sampling rate for production."),
    ("User reported: login works but session drops after exactly 15 minutes.", "That's your access token expiry (15min). Check if refresh token flow is working — client should auto-refresh before expiry."),
    ("Bug confirmed: refresh endpoint was returning new access token but not extending the refresh token.", "Token rotation issue. On refresh: issue new access token AND new refresh token. Invalidate the old refresh token."),
    ("Fixed. Now refresh rotates both tokens. Added refresh token family tracking.", "Token family tracking prevents replay attacks. If old refresh token is reused, invalidate the entire family."),
    ("Load testing with locust: 500 concurrent users, 99th percentile at 200ms.", "Solid numbers. Bottleneck will likely be DB connections. Ensure pool size matches expected concurrency."),
    ("DB pool at 20 connections. Getting occasional 'too many connections' with 500 users.", "Increase pool to 50 with overflow=10. Also consider PgBouncer for connection pooling at the DB level."),
    ("PgBouncer solved it. Now stable at 500 concurrent, p99 stays under 150ms.", "Much better. PgBouncer's connection multiplexing is ideal for high-concurrency Python async apps."),
    ("Final concern: security audit found no CORS configuration.", "Critical fix: add CORSMiddleware with explicit allowed origins. Never use allow_origins=['*'] in production."),
    ("Added CORS with specific origins. Also added security headers middleware.", "Good. Key headers: X-Content-Type-Options, X-Frame-Options, Strict-Transport-Security, Content-Security-Policy."),
    ("Production launch tomorrow. Checklist?", "Launch checklist: 1) HTTPS enforced, 2) Rate limiting active, 3) Monitoring/alerts, 4) Backup DB, 5) Rollback plan, 6) On-call rotation."),
    ("Launched! 2000 users first day, zero errors.", "Congratulations! Monitor closely for 48 hours. Watch: error rates, latency percentiles, DB connection usage, memory growth."),
    ("Day 2: memory leak detected. RSS growing 50MB/hour.", "Async leak pattern. Common causes: unclosed DB sessions, accumulated middleware state, response object retention. Profile with tracemalloc."),
    ("Found it: forgot to close async generators in a streaming endpoint.", "Async generators must be properly closed. Use 'async with' or try/finally. Add cleanup in shutdown event handler."),
]

for user_msg, assistant_msg in topics:
    conversation_turns.append(("user", user_msg))
    conversation_turns.append(("assistant", assistant_msg))

print(f"✓ Prepared {len(topics)} turn pairs ({len(conversation_turns)} messages) for stress test")

In [ ]:
# Run the 30-turn conversation through all 4 strategies
strategies = {
    "FullHistory": FullHistoryMemory("You are a backend development assistant."),
    "SlidingWindow(5)": SlidingWindowMemory("You are a backend development assistant.", window_size=5),
    "SlidingWindow(10)": SlidingWindowMemory("You are a backend development assistant.", window_size=10),
    "ImportanceWeighted(3000)": ImportanceWeightedMemory("You are a backend development assistant.", max_tokens=3000),
}

# Track tokens at each step
results = {name: [] for name in strategies}

for i in range(0, len(conversation_turns), 2):
    user_msg = conversation_turns[i]
    assistant_msg = conversation_turns[i + 1]
    turn_num = i // 2 + 1

    for name, strategy in strategies.items():
        strategy.add_message(user_msg[0], user_msg[1])
        strategy.add_message(assistant_msg[0], assistant_msg[1])
        results[name].append(strategy.token_count())

# Display results
print("30-Turn Stress Test — Token Usage Comparison")
print("=" * 80)
header = f"{'Turn':>5}"
for name in strategies:
    short_name = name.split("(")[0][:12]
    header += f" {short_name:>12}"
print(header)
print("-" * 80)

for turn in [1, 5, 10, 15, 20, 25, 30]:
    idx = turn - 1
    row = f"{turn:5d}"
    for name in strategies:
        tokens = results[name][idx]
        row += f" {tokens:12,}"
    print(row)

print("-" * 80)
print("\n📊 Summary:")
for name in strategies:
    final = results[name][-1]
    peak = max(results[name])
    print(f"  {name:<30} Final: {final:6,} tokens  Peak: {peak:6,} tokens")

## 8. Working Memory Pattern

Beyond conversation history, agents benefit from a **structured scratchpad** — a working memory where they can explicitly store and retrieve key findings.

This is inspired by human working memory: we don't just remember conversations, we maintain a mental "whiteboard" of current facts, goals, and intermediate results.

The pattern:
- Agent has a `working_memory` dict
- Tools `store(key, value)` and `recall(key)` for explicit memory management
- Working memory is included in the system prompt at each step

In [ ]:
class WorkingMemory:
    """Structured scratchpad for agent key findings and state."""

    def __init__(self):
        self.store = {}
        self.access_log = []  # track reads and writes

    def write(self, key, value):
        """Store a key-value pair."""
        self.store[key] = {
            "value": value,
            "written_at": time.time(),
            "access_count": 0
        }
        self.access_log.append(("write", key, time.time()))
        return f"Stored: {key} = {value}"

    def read(self, key):
        """Read a value by key."""
        if key not in self.store:
            return f"Key '{key}' not found. Available keys: {list(self.store.keys())}"
        entry = self.store[key]
        entry["access_count"] += 1
        self.access_log.append(("read", key, time.time()))
        return entry["value"]

    def list_keys(self):
        """List all stored keys with previews."""
        if not self.store:
            return "Working memory is empty."
        lines = ["Working Memory Contents:"]
        for key, entry in self.store.items():
            preview = str(entry["value"])[:80]
            lines.append(f"  • {key}: {preview}")
        return "\n".join(lines)

    def delete(self, key):
        """Remove a key from working memory."""
        if key in self.store:
            del self.store[key]
            return f"Deleted: {key}"
        return f"Key '{key}' not found."

    def to_context_string(self):
        """Format working memory for inclusion in system prompt."""
        if not self.store:
            return "Working memory: (empty)"
        lines = ["Current working memory:"]
        for key, entry in self.store.items():
            lines.append(f"  {key}: {entry['value']}")
        return "\n".join(lines)

    def __repr__(self):
        return f"WorkingMemory({len(self.store)} entries)"


# Demonstrate working memory
wm = WorkingMemory()

# Simulate an agent storing findings during a research task
wm.write("project_type", "FastAPI REST API with PostgreSQL")
wm.write("auth_method", "JWT with bcrypt (cost 12)")
wm.write("current_issue", "401 errors on protected endpoints")
wm.write("root_cause", "Different JWT secrets between creation and validation")
wm.write("performance", "p99 latency: 150ms at 500 concurrent users")

print(wm)
print()
print(wm.to_context_string())
print()

# Agent can recall specific facts
print("\nRecall examples:")
print(f"  Auth method: {wm.read('auth_method')}")
print(f"  Root cause: {wm.read('root_cause')}")
print(f"  Missing key: {wm.read('deadline')}")

In [ ]:
# Agent with Working Memory — Full Integration
class WorkingMemoryAgent:
    """ReAct agent enhanced with a working memory scratchpad."""

    def __init__(self, system_prompt):
        self.system_prompt = system_prompt
        self.conversation = SlidingWindowMemory(system_prompt, window_size=8)
        self.working_memory = WorkingMemory()
        self.tools = {
            "store": self._tool_store,
            "recall": self._tool_recall,
            "list_memory": self._tool_list,
        }

    def _tool_store(self, args):
        """store(key, value) — save a finding to working memory."""
        if "," not in args:
            return "Error: use format store(key, value)"
        key, value = args.split(",", 1)
        return self.working_memory.write(key.strip(), value.strip())

    def _tool_recall(self, args):
        """recall(key) — retrieve a value from working memory."""
        return self.working_memory.read(args.strip())

    def _tool_list(self, args):
        """list_memory() — show all stored keys."""
        return self.working_memory.list_keys()

    def _build_system_prompt(self):
        """Build system prompt with current working memory."""
        wm_context = self.working_memory.to_context_string()
        return f"""{self.system_prompt}

{wm_context}

You have access to these memory tools:
- store(key, value) — save an important finding
- recall(key) — retrieve a stored finding
- list_memory() — see all stored findings

When you discover important facts, store them. When you need context, recall them.
Format tool calls as: Action: tool_name(arguments)"""

    def step(self, user_input):
        """Process one user message and generate response."""
        self.conversation.add_message("user", user_input)

        # Build context with updated working memory
        system = self._build_system_prompt()
        messages = [{"role": "system", "content": system}]
        messages.extend(self.conversation.messages[-16:])  # recent messages

        response = generate(messages, max_new_tokens=300, temperature=0.7)

        # Check for tool calls in response
        if "Action:" in response:
            lines = response.split("\n")
            for line in lines:
                if line.strip().startswith("Action:"):
                    action = line.strip()[7:].strip()
                    # Parse tool_name(args)
                    if "(" in action and action.endswith(")"):
                        tool_name = action[:action.index("(")]
                        tool_args = action[action.index("(")+1:-1]
                        if tool_name in self.tools:
                            result = self.tools[tool_name](tool_args)
                            response += f"\nObservation: {result}"

        self.conversation.add_message("assistant", response)
        return response

# Demonstrate the working memory agent
wm_agent = WorkingMemoryAgent("You are a project planning assistant. Store key decisions and facts.")

# Simulate a multi-turn planning session
turns = [
    "We're building a mobile app for fitness tracking. The target launch date is March 2025.",
    "The tech stack decision: React Native for cross-platform, Firebase for backend.",
    "What tech stack did we choose?",
    "Team size is 4 developers, 1 designer, 1 PM. Sprint length is 2 weeks.",
    "Remind me of the project details so far.",
]

print("Working Memory Agent — Multi-Turn Demo")
print("=" * 60)

for turn_input in turns:
    print(f"\n👤 User: {turn_input}")
    response = wm_agent.step(turn_input)
    print(f"🤖 Agent: {response[:400]}")
    if wm_agent.working_memory.store:
        print(f"   📋 WM: {list(wm_agent.working_memory.store.keys())}")

## 9. Strategy Comparison

Let's consolidate everything into a comparison table.

In [ ]:
# Comprehensive comparison
print("=" * 90)
print("SHORT-TERM MEMORY STRATEGY COMPARISON")
print("=" * 90)

comparison = [
    {
        "strategy": "Full History",
        "token_bound": "Unbounded",
        "retention": "Perfect",
        "complexity": "O(1) add",
        "pros": "Zero information loss",
        "cons": "Exceeds context window",
        "best_for": "Short conversations (<20 turns)"
    },
    {
        "strategy": "Sliding Window",
        "token_bound": "Fixed",
        "retention": "Recent only",
        "complexity": "O(1) add",
        "pros": "Simple, predictable",
        "cons": "Hard cutoff loses old info",
        "best_for": "Stateless or recent-context tasks"
    },
    {
        "strategy": "Summarization",
        "token_bound": "Bounded",
        "retention": "Compressed old + full recent",
        "complexity": "O(n) on trigger",
        "pros": "Retains key info from all turns",
        "cons": "Summarization cost + info loss",
        "best_for": "Long sessions needing full context"
    },
    {
        "strategy": "Importance-Weighted",
        "token_bound": "Fixed budget",
        "retention": "Highest-value messages",
        "complexity": "O(n log n) select",
        "pros": "Prioritizes valuable content",
        "cons": "Heuristic may misjudge importance",
        "best_for": "Mixed-importance conversations"
    },
]

for c in comparison:
    print(f"\n📌 {c['strategy']}")
    print(f"   Token bound:   {c['token_bound']}")
    print(f"   Retention:     {c['retention']}")
    print(f"   Complexity:    {c['complexity']}")
    print(f"   ✅ Pros:       {c['pros']}")
    print(f"   ❌ Cons:       {c['cons']}")
    print(f"   🎯 Best for:   {c['best_for']}")

print("\n" + "=" * 90)
print("DECISION GUIDE")
print("=" * 90)
print("""
  Short conversations (<20 turns)?     → Full History
  Only recent context matters?          → Sliding Window
  Need to reference old decisions?      → Summarization
  Mixed-importance, token-constrained?  → Importance-Weighted
  Production system?                    → Summarization + Working Memory
""")

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** extend the agent with a new tool. Document what changes and why.

**Exercise 2 — Build:** add error handling for a failure mode discussed in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** trace through the agent loop manually and predict the output before running.

## 10. Key Takeaways

### What We Built
1. **FullHistoryMemory** — perfect retention but unbounded growth
2. **SlidingWindowMemory** — bounded and simple but loses old context
3. **SummarizationMemory** — compresses old context using the LLM itself
4. **ImportanceWeightedMemory** — prioritizes high-value messages within a budget
5. **MemoryManager** — unified controller with token tracking and automatic compression
6. **WorkingMemory** — structured scratchpad for explicit knowledge management

### Key Insights

- **There is no perfect memory strategy** — each trades off retention vs. efficiency
- **Summarization is the most practical** for long conversations but costs LLM calls
- **Working memory complements conversation memory** — explicit storage beats implicit context
- **Token counting is essential** — you can't manage what you don't measure
- **Production agents combine strategies** — summarized history + working memory + importance filtering

### What's Next
- **Notebook 11**: Long-term memory with vector stores — memory that persists across sessions
- **Notebook 12**: Knowledge graph memory — structured relationships between facts
- **Notebook 13**: Advanced tool design — building robust tools for agents

### References
- Sumers et al. (2024) — *"Cognitive Architectures for Language Agents"* (CoALA framework)
- Park et al. (2023) — *"Generative Agents: Interactive Simulacra of Human Behavior"*
- Packer et al. (2023) — *"MemGPT: Towards LLMs as Operating Systems"*